In [2]:
import numpy as np
import heapq as hq

In [8]:
# class that implements tableau and necessary operations
# Reference -Improved Simulation of Stabilizer Circuits by Scott Aaronson & Daniel Gottesman
class Cirq_Tableau:
    
    def __init__(
        self,
        pauli_word: list[str] = None  
    ):
        if pauli_word is None or len(pauli_word) == 0:
            self._column_num = None
            self._row_num = None

            self._ss, self._xs, self._zs = None, None, None
        else:
            if "-" in pauli_word[0]:
                self._column_num = len(pauli_word[0][1:]) 
            else:
                self._column_num = len(pauli_word[0])
            self._row_num = len(pauli_word)

            self._ss = self.create_sign(pauli_word)
            self._xs, self._zs = self.create_tab(pauli_word)

    # setters & getters 
    @property
    def ss(self) -> np.ndarray:
        return self._ss

    @ss.setter 
    def ss(self, new_ss: np.ndarray):
        self._ss = new_ss
        
    @property
    def xs(self) -> np.ndarray:
        return self._xs

    @xs.setter 
    def xs(self, new_xs: np.ndarray):
        self._xs = new_xs
        
    @property
    def zs(self) -> np.ndarray:
        return self._zs

    @zs.setter 
    def zs(self, new_zs: np.ndarray):
        self._zs = new_zs
        
    @property
    def column_num(self) -> int:
        return self._column_num

    @column_num.setter 
    def column_num(self, new_num: int):
        self._column_num = new_num
    
    @property
    def row_num(self) -> int:
        return self._row_num

    @row_num.setter 
    def row_num(self, new_num: int):
        self._row_num = new_num

    # functions that create parts of tableau
    def create_sign(self, pauli_word: list[str]):
        temp_ss = np.zeros((self.row_num), dtype=int)
        for pauli_string in pauli_word:
            if "-" in pauli_string:
                index = pauli_word.index(pauli_string)
                pauli_word[index] = pauli_string.replace("-", "")
                #print(pauli_word)
                temp_ss[index] = 1
        return temp_ss
        
    def create_tab(self, pauli_word: list[str]):
        temp_x = np.zeros((self.row_num, self.column_num), dtype=int)
        temp_z = np.zeros((self.row_num, self.column_num), dtype=int)
        for i, pauli_string in enumerate(pauli_word):
            for j, pauli in enumerate(pauli_string):
                if pauli == "X" or pauli == "Y":
                    temp_x[i][j] = 1
                if pauli == "Z" or pauli == "Y":
                    temp_z[i][j] = 1
        return temp_x, temp_z

    # Clifford operations on tableau. 
    def apply_H(self,column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.xs[:, column], self.zs[:, column] = self.zs[:, column].copy(), self.xs[:, column].copy()

    def apply_S(self, column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.zs[:, column] = self.xs[:, column] ^ self.zs[:, column]

    def apply_CX(self, control: int, target: int):
        self.ss ^= (
            (self.xs[:, control] & self.zs[:, target])
            &(~(self.xs[:, target] ^ self.zs[:, control]))
        )
        self.xs[:, target] ^= self.xs[:, control]
        self.zs[:, control] ^= self.zs[:, target]

    # class operations necessary for comparisons and equating
    def copy(self):
        new_tab = Cirq_Tableau()
        new_tab.column_num = self.column_num
        new_tab.row_num = self.row_num
        new_tab.ss = self.ss.copy()
        new_tab.zs = self.zs.copy()
        new_tab.xs = self.xs.copy()
        return new_tab
    
    def __eq__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented  
        return (
            self.column_num == other.column_num
            and self.row_num == other.row_num
            and np.array_equal(self.ss, other.ss)
            and np.array_equal(self.xs, other.xs)
            and np.array_equal(self.zs, other.zs)
        )
    
    def return_string(self):
        string = ''
        for i in range(self.row_num):
            if self.ss[i]:
                string += "-"  

            for j in range(self.column_num):
                if self.xs[i][j] and not self.zs[i][j]:
                    string += "X"
                elif not self.xs[i][j] and self.zs[i][j]:
                    string += "Z"
                elif self.xs[i][j] and self.zs[i][j]:
                    string += "Y"
                else:
                    string += "I"
            if i < self.row_num - 1:
                string += "\n" 

        return string
    
    def __copy__(self):
        return self.copy()
        

    def __str__(self) -> str:
        ss = np.expand_dims(self.ss, axis = 1)
        xz = np.concatenate((self.xs, self.zs, ss), axis=1)
        return str(xz)

    def __hash__(self) -> int:
        return hash(self.zs.tobytes() + self.xs.tobytes() + self.ss.tobytes())

    def __lt__(self, other):
        return self.row_num < other.row_num

In [ ]:
def subroutine(tableau, ndx):
    tab = tableau.copy()
    path = []
    while True:
        

In [9]:
def singe_pauli_subroutine(tableau, ndx):
    tab = tableau.copy()
    path = []
    while True:
        pair = None
        benefit = -float('inf')
        reduced = 0
        added = 0
        
        

        for i in range(tab.column_num):
            if i != tab.column_num -1:
                for j in range(i+1, tab.column_num):
                    if (~(tab.xs[ndx][i]) & tab.zs[ndx][i] & tab.zs[ndx][j]) | (tab.xs[ndx][i] & tab.xs[ndx][j] & ~(tab.zs[ndx][j])):
                        reduce = sum((~(tab.xs[:, i]) & tab.zs[:, i] & tab.zs[:, j]) | (tab.xs[:, i] & tab.xs[:, j] & ~(tab.zs[:, j])))
                        addition = sum((~(tab.xs[:, i]) & ~(tab.zs[:, i]) & tab.zs[:, j]) | (tab.xs[:, i] & ~(tab.xs[:, j]) & ~(tab.zs[:, j])))
                        if (reduce - addition) > benefit:
                            benefit = reduce - addition
                            reduced = reduce
                            added = addition
                            pair = (i, j)
                        elif (reduce - addition) == benefit:
                            if addition < added:
                                benefit = reduce - addition
                                reduced = reduce 
                                added = addition
                                pair = (i, j)
                       
                    if (~(tab.xs[ndx][j]) & tab.zs[ndx][j] & tab.zs[ndx][i]) | (tab.xs[ndx][j] & tab.xs[ndx][i] & ~(tab.zs[ndx][i])):
                        reduce = sum((~(tab.xs[:, j]) & tab.zs[:, j] & tab.zs[:, i]) | (tab.xs[:, j] & tab.xs[:, i] & ~(tab.zs[:, i])))
                        addition = sum((~(tab.xs[:, j]) & ~(tab.zs[:, j]) & tab.zs[:, i]) | (tab.xs[:, j] & ~(tab.xs[:, i]) & ~(tab.zs[:, i])))
                        if (reduce - addition) > benefit:
                            benefit = reduce - addition
                            reduced = reduce
                            added = addition
                            pair = (j, i)
                        elif (reduce - addition) == benefit:
                            if addition < added:
                                benefit = reduce - addition
                                reduced = reduce 
                                added = addition
                                pair = (j, i)
                        
                        
        
        
        
        
        if pair != None:
            operation = pair
            tab.apply_CX(operation[0], operation[1])
            path.append(("CX",operation[0], operation[1]))
            
        if sum(tab.xs[ndx] | tab.zs[ndx]) == 1:
            break    
            
        reducible = sum(tab.xs[ndx] ^ tab.zs[ndx])
        
        if  reducible == 0:
            for i in range(tab.column_num):
                if(tab.xs[ndx][i] & tab.zs[ndx][i]):
                    tab.apply_S(i)
                    path.append(("S",i))
                    break
        elif reducible == 2:
            for i in range(tab.column_num):
                if(~(tab.xs[ndx][i]) & tab.zs[ndx][i]):
                    tab.apply_H(i)
                    path.append(("H",i))
                    break

    sum_array = tab.xs | tab.zs
    weight_sum = sum([int(sum(x)) for x in sum_array if sum(x) != 1])
    implemented = sum([1 for x in sum_array if sum(x) == 1])
    return tab, weight_sum, path, implemented

In [10]:
def prune(tableau: Cirq_Tableau):
    '''
    Function to remove rows in a tableau with only a single nonidentity operation
    
    :param tableau: Cirq_Tableau to act on
    '''
    prn_ndxs = [] # list to hold all row indices to be pruned
    
    tab = tableau.copy() # copy of tableau to work on

    single_q = [] # loist of single qubit operations 
    
    # extracts x, z and sign arrays
    x, z, s = tab.xs, tab.zs, tab.ss

    # bitwise ORs x and z arrays in tableau to create array where nonidentity operation indices have 1 in them
    weight_array = x | z

    # loop that checks through each row to find out if it has a Pauli weight of 1
    for r_ndx in range(len(weight_array)):
        if sum(weight_array[r_ndx])  == 1:

            # appends index to be pruned if Pauli weight == 1
            prn_ndxs.append(r_ndx)

            # checks what type of Pauli is on that index X, Y or Z and appends to path with action weight of zero 
            if sum(z[r_ndx]) == 0:
                if s[r_ndx] == 0:
                    single_q.append(("X", int(np.argmax(x[r_ndx] == 1))))
                else:
                    single_q.append(("-X", int(np.argmax(x[r_ndx] == 1))))
            elif sum(x[r_ndx]) == 0:
                if s[r_ndx] == 0:
                    single_q.append(("Z", int(np.argmax(z[r_ndx] == 1))))
                else:
                    single_q.append(("-Z", int(np.argmax(z[r_ndx] == 1))))
            else:
                if s[r_ndx] == 0:
                    single_q.append(("Y", int(np.argmax(x[r_ndx] == 1))))
                else:
                    single_q.append(("-Y", int(np.argmax(x[r_ndx] == 1))))
                    
        elif sum(weight_array[r_ndx])  == 0: # elif removes fully I lines 
            x, z, s = np.delete(x, r_ndx, axis=0), np.delete(z, r_ndx, axis=0), np.delete(s, r_ndx)
            
    # prunes rows 
    x, z, s = np.delete(x, prn_ndxs, axis=0), np.delete(z, prn_ndxs, axis=0), np.delete(s, prn_ndxs)
    column_num = len(x[0]) if x.size != 0 else 0 
    row_num = len(x)

    # updates tableau and returns it 
    tab.xs, tab.zs, tab.ss, tab.column_num, tab.row_num = x, z, s, column_num,row_num
    return tab, single_q
    

In [11]:
word = ["XZXZXZIIX","ZYZZIYZYZ", "YYZYXIXXZ", "ZYIXXZZZZ", "YXYZIXXIY", "YXZZYXYIX" ]

In [15]:
 tb, w_sum, p, imp= singe_pauli_subroutine(Cirq_Tableau(word), 4)
print(tb.return_string())
print(w_sum)
print(p)
print(imp)

IYZZXYXIX
IYIIIYZYI
-ZYIXXIXXI
-YYIYXYZZI
IIIIIIIIY
-XIXIYIYIX
29
[('CX', 8, 1), ('CX', 3, 0), ('CX', 0, 5), ('CX', 2, 6), ('S', 0), ('CX', 2, 0), ('S', 2), ('CX', 8, 2)]
1


In [13]:
def tree_search(start):
    full_path =[]
    node = Cirq_Tableau(start)
    curr_path = None
    curr_node = None
    w_sum = float('inf')
    implemented = float('inf')
    while True:
        
        if node.row_num == 0:
            break
            
        for i in range(node.row_num):
            
            new_node, weight, node_path, node_imp = singe_pauli_subroutine(node, i)
            if weight < w_sum:
                curr_node, single = prune(new_node)
                node_path.extend(single)
                curr_path = node_path
                w_sum = weight
                implemented = node_imp
            elif weight == w_sum:
                if node.row_num > new_node.row_num - node_imp:
                    curr_node, single = prune(new_node)
                    node_path.extend(single)
                    curr_path = node_path
                    w_sum = weight
                    implemented == node_imp
                    
        node = curr_node
        full_path += curr_path

    return full_path

In [14]:
tree_search(word)

[('CX', 8, 1),
 ('CX', 3, 0),
 ('CX', 0, 5),
 ('CX', 2, 6),
 ('S', 0),
 ('CX', 2, 0),
 ('S', 2),
 ('CX', 8, 2),
 ('Y', 8),
 ('CX', 6, 4),
 ('CX', 1, 6),
 ('CX', 1, 3),
 ('H', 0),
 ('CX', 0, 7),
 ('CX', 1, 0),
 ('-Y', 1),
 ('CX', 3, 5),
 ('CX', 5, 4),
 ('S', 1),
 ('CX', 5, 1),
 ('S', 5),
 ('CX', 7, 5),
 ('S', 6),
 ('CX', 7, 6),
 ('-Y', 7),
 ('CX', 0, 1),
 ('CX', 1, 4),
 ('CX', 2, 8),
 ('CX', 4, 6),
 ('H', 7),
 ('CX', 6, 2),
 ('CX', 6, 7),
 ('Y', 6),
 ('CX', 6, 0),
 ('CX', 7, 3),
 ('S', 0),
 ('CX', 3, 0),
 ('S', 3),
 ('CX', 5, 3),
 ('-Y', 5),
 ('CX', 1, 2),
 ('CX', 2, 3),
 ('H', 5),
 ('CX', 3, 5),
 ('CX', 3, 8),
 ('Y', 3)]